# 05 — Priprema podataka za mašinsko učenje

Ova sveska transformiše sirove trening i test skupove u oblik pogodan za
algoritme mašinskog učenja. Sve odluke se donose **isključivo na osnovu
trening skupa**, a iste transformacije se dosledno primenjuju i na test skup.

**Ovde cemo primeniti nekoliko vaznih koncepta:**

1. **Feature engineering** — spajanje kategorija, konverzija tipova, kreiranje izvedenih atributa ,itd.
2. **Enkodiranje kategorija** — pretvaranje tekstualnih vrednosti u brojeve
   pomoću tehnika poput:
      - Label
      - Ordinal
      - One-Hot enkodiranja

Nakon toga treba da sacuvamo pripremljene podatake


## 1. Učitavanje trening i test skupova

Učitavamo sva četiri objekta koje smo sačuvali u svesci `03`:
`X_train`, `X_test`, `y_train`, `y_test`.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

putanja_processed = "../data/processed/"

X_train = pd.read_csv(putanja_processed + "X_train.csv")
X_test = pd.read_csv(putanja_processed + "X_test.csv")
y_train = pd.read_csv(putanja_processed + "y_train.csv")
y_test = pd.read_csv(putanja_processed + "y_test.csv")

# y_train i y_test su učitani kao DataFrame sa jednom kolonom.
# Pretvaramo ih u Series (1D niz) jer je to standardni format za ciljnu promenljivu.
# .squeeze() automatski pretvara jednokolonski DataFrame u Series.
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print("Učitani podaci:")
print(f"  X_train: {X_train.shape[0]} × {X_train.shape[1]}")
print(f"  X_test:  {X_test.shape[0]} × {X_test.shape[1]}")
print(f"  y_train: {y_train.shape[0]}")
print(f"  y_test:  {y_test.shape[0]}")

Učitani podaci:
  X_train: 5625 × 19
  X_test:  1407 × 19
  y_train: 5625
  y_test:  1407


## 2. Feature engineering

Pre samog enkodiranja, primenjujemo transformacije

### 2.1 Spajanje "No internet service" i "No"

Šest kolona (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`,
`TechSupport`, `StreamingTV`, `StreamingMovies`) ima tri moguće vrednosti:
`Yes`, `No`, `No internet service`.

Sa poslovne strane, „korisnik nema tu uslugu" je isto stanje bez obzira
da li je razlog „ima internet ali nije uzeo tu opciju" ili „uopšte nema
internet". Zato svodimo `No internet service` na `No`.

Slično, kolona `MultipleLines` ima `No phone service` — koju svodimo na `No`.

In [2]:
kolone_no_internet = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]


for kolona in kolone_no_internet:
    X_train[kolona] = X_train[kolona].replace("No internet service", "No")
    X_test[kolona] = X_test[kolona].replace("No internet service", "No")


X_train["MultipleLines"] = X_train["MultipleLines"].replace("No phone service", "No")
X_test["MultipleLines"] = X_test["MultipleLines"].replace("No phone service", "No")

# Provera - sada te kolone treba da imaju samo "Yes" i "No".
print("Vrednosti nakon spajanja:")
for kolona in kolone_no_internet + ["MultipleLines"]:
    print(f"  {kolona}: {sorted(X_train[kolona].unique())}")

Vrednosti nakon spajanja:
  OnlineSecurity: ['No', 'Yes']
  OnlineBackup: ['No', 'Yes']
  DeviceProtection: ['No', 'Yes']
  TechSupport: ['No', 'Yes']
  StreamingTV: ['No', 'Yes']
  StreamingMovies: ['No', 'Yes']
  MultipleLines: ['No', 'Yes']


### 2.2 Konverzija `SeniorCitizen` u tekstualnu binarnu kategoriju

Kolona `SeniorCitizen` je zapisana kao broj (0/1), ali konceptualno je
binarna kategorija. Konvertujemo je u `No`/`Yes` radi konzistentnosti sa
ostalim binarnim kolonama.

Ovo pomaže kod enkodiranja u sledećem koraku — sve binarne kolone
možemo tretirati istom logikom.

In [3]:
#proveravamo sa if-om da bi obezbedili sigurnost od veceg broja pokretanja
if X_train["SeniorCitizen"].dtype != "object":
    mapiranje_senior = {0: "No", 1: "Yes"}
    X_train["SeniorCitizen"] = X_train["SeniorCitizen"].map(mapiranje_senior)
    X_test["SeniorCitizen"] = X_test["SeniorCitizen"].map(mapiranje_senior)
print("SeniorCitizen vrednosti nakon konverzije:")
print(X_train["SeniorCitizen"].value_counts())

SeniorCitizen vrednosti nakon konverzije:
SeniorCitizen
No     4715
Yes     910
Name: count, dtype: int64


## 3. Enkodiranje kategorijskih promenljivih

Sada pretvaramo sve tekstualne kolone u brojeve. Koristimo **tri različite
metode**, biramo prema tipu kategorije:

- **Label Encoding (0/1)** — za binarne kolone (`Yes`/`No`)
- **Ordinal Encoding** — za jedinu ordinalnu kolonu (`Contract`)
- **One-Hot Encoding** — za nominalne kolone sa više od 2 vrednosti

### Pregled kolona po metodi:

| Metoda | Kolone |
|---|---|
| Label (0/1) | `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaperlessBilling` |
| Ordinal | `Contract` (Month-to-month < One year < Two year) |
| One-Hot | `InternetService`, `PaymentMethod` |

Numeričke kolone (`tenure`, `MonthlyCharges`, `TotalCharges`) ostaju
nepromenjene jer su već brojevi.

### 3.1 Label Encoding za binarne kolone

Za sve binarne kolone (`Yes`/`No`) primenjujemo isto mapiranje: `Yes → 1`,
`No → 0`

In [4]:
binarne_kolone = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling"
]

# kolona "gender" ima "Male"/"Female", ne "Yes"/"No". Zbog toga resavamo je zasebno pre opšteg mapiranja.
mapiranje_gender = {"Male": 1, "Female": 0}
X_train["gender"] = X_train["gender"].map(mapiranje_gender)
X_test["gender"] = X_test["gender"].map(mapiranje_gender)


# Opšte mapiranje za ostale binarne kolone (Yes/No).
mapiranje_yes_no = {"Yes": 1, "No": 0}
for kolona in binarne_kolone:
    if kolona == "gender":
        continue  
    X_train[kolona] = X_train[kolona].map(mapiranje_yes_no)
    X_test[kolona] = X_test[kolona].map(mapiranje_yes_no)


print("Prve 3 vrste nakon Label enkodiranja binarnih kolona:")
X_train[binarne_kolone].head(3)

Prve 3 vrste nakon Label enkodiranja binarnih kolona:


,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling
0,1,0,1,1,1,1,1,1,1,1,0,0,0
1,1,0,0,0,0,0,0,0,1,1,0,0,0
2,0,0,1,0,1,1,0,1,1,1,0,0,0


### 3.2 Ordinal Encoding za `Contract`

Kolona `Contract` ima tri vrednosti sa **prirodnim redosledom**:
- `Month-to-month` (najkraći, najfleksibilniji, ali najveći rizik od odlaska)
- `One year` (srednji)
- `Two year` (najduži, najveća lojalnost)

Mapiramo ih redosledom `0 → 1 → 2`, čime zadržavamo informaciju o
progresiji dužine ugovora.

**Zašto ne One-Hot:** iako bi i One-Hot radio, Ordinal je bolji jer čuva
informaciju da su vrednosti povezane redosledom. Modeli bazirani na stablima
(Random Forest, XGBoost) posebno dobro koriste ovu informaciju.

In [5]:
mapiranje_contract = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2
}

X_train["Contract"] = X_train["Contract"].map(mapiranje_contract)
X_test["Contract"] = X_test["Contract"].map(mapiranje_contract)

# Provera.
print("Raspodela Contract nakon Ordinal enkodiranja:")
print(X_train["Contract"].value_counts().sort_index())

Raspodela Contract nakon Ordinal enkodiranja:
Contract
0    3085
1    1182
2    1358
Name: count, dtype: int64


### 3.3 One-Hot Encoding za nominalne kolone

Preostale su dve nominalne kolone bez prirodnog redosleda:
- **`InternetService`** — `DSL`, `Fiber optic`, `No` (3 vrednosti)
- **`PaymentMethod`** — `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)` (4 vrednosti)

Za njih koristimo **One-Hot Encoding**: svaka kategorija postaje **nova
kolona** sa vrednostima 0 ili 1.

**Važna napomena:** koristimo `drop_first=True` da izbacimo jednu kolonu
po svakoj promenljivoj (npr. za `InternetService` izbacimo `InternetService_DSL`).
Razlog je što je ta informacija **redundantna** — ako model zna da nije
`Fiber optic` i nije `No`, mora biti `DSL`. Ovim izbegavamo **savršenu
kolinearnost** koja pravi probleme kod linearnih modela.

In [6]:
nominalne_kolone = ["InternetService", "PaymentMethod"]

# primenjujemo istu transformaciju na oba skupa.
# funkcija get_dummies vraca True/False pa mi treba da konvertujemo u int (0/1) za konzistentnost sa ostalim binarnim kolonama.
X_train = pd.get_dummies(X_train, columns=nominalne_kolone, drop_first=True)
X_test = pd.get_dummies(X_test, columns=nominalne_kolone, drop_first=True)


bool_kolone = X_train.select_dtypes(include=["bool"]).columns
X_train[bool_kolone] = X_train[bool_kolone].astype(int)
X_test[bool_kolone] = X_test[bool_kolone].astype(int)

# Provera - trebalo bi da imamo nove kolone.
print("Nove kolone nakon One-Hot Encoding-a:")
nove_kolone = [k for k in X_train.columns if k.startswith("InternetService_") or k.startswith("PaymentMethod_")]
print(nove_kolone)

Nove kolone nakon One-Hot Encoding-a:
['InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


### 3.4 Provera konzistentnosti kolona train i test skupa

Nakon One-Hot Encoding-a, moramo proveriti da train i test imaju
**identičan skup kolona**. Ako neka kategorija postoji u trainu a ne u
testu (ili obrnuto), broj kolona se ne poklapa i modeli će pući.

In [7]:
# Poredimo skupove kolona.
kolone_train = set(X_train.columns)
kolone_test = set(X_test.columns)

samo_u_train = kolone_train - kolone_test
samo_u_test = kolone_test - kolone_train

print(f"Broj kolona u X_train: {len(kolone_train)}")
print(f"Broj kolona u X_test:  {len(kolone_test)}")

if samo_u_train or samo_u_test:
    print("\nUPOZORENJE - kolone se ne poklapaju:")
    if samo_u_train:
        print(f"  Samo u train: {samo_u_train}")
    if samo_u_test:
        print(f"  Samo u test: {samo_u_test}")
else:
    print("\nSve kolone se poklapaju - train i test su konzistentni.")

Broj kolona u X_train: 22
Broj kolona u X_test:  22

Sve kolone se poklapaju - train i test su konzistentni.


### 3.5 Enkodiranje ciljne promenljive `y`

Ciljna promenljiva `Churn` je binarna sa vrednostima `Yes` (napustio) i
`No` (ostao). Mapiramo je u 1/0 na način koji ima jasno tumačenje:

- `Yes` (churn) → **1** (pozitivna klasa — ono što predviđamo)
- `No` (ostao) → **0** (negativna klasa)

**Napomena:** u klasifikaciji je konvencija da je ono što nas zanima
označeno sa 1 (pozitivna klasa). Za nas je to churn, jer je cilj modela
da identifikuje korisnike koji će otići.

In [8]:
# Mapiranje ciljne promenljive.
mapiranje_churn = {"Yes": 1, "No": 0}

y_train = y_train.map(mapiranje_churn)
y_test = y_test.map(mapiranje_churn)

# Provera.
print("Raspodela y_train nakon enkodiranja:")
print(y_train.value_counts())
print(f"\nProcenat pozitivne klase (Yes/1): {100 * y_train.mean():.2f}%")

Raspodela y_train nakon enkodiranja:
Churn
0    4130
1    1495
Name: count, dtype: int64

Procenat pozitivne klase (Yes/1): 26.58%


## 4. Provera finalnog stanja pripremljenih podataka

Pre čuvanja, verifikujemo da su podaci potpuno spremni za mašinsko učenje:

- Sve kolone su numeričke
- Nema nedostajućih vrednosti
- Broj kolona train i test skupa je identičan
- Dimenzije se poklapaju sa očekivanjem

In [ ]:
# 1. Provera tipova svih kolona - sve treba da budu numeričke.
print("=" * 60)
print("PROVERA TIPOVA KOLONA")
print("=" * 60)
print("Tipovi kolona u X_train:")
print(X_train.dtypes.value_counts())


object_kolone = X_train.select_dtypes(include=["object"]).columns
if len(object_kolone) > 0:
    print(f"\nUPOZORENJE - preostale tekstualne kolone: {list(object_kolone)}")
else:
    print("\nSve kolone su numeričke.")

# 2. Provera nedostajućih vrednosti.
print("\n" + "=" * 60)
print("PROVERA NEDOSTAJUĆIH VREDNOSTI")
print("=" * 60)

na_train = X_train.isnull().sum().sum()
na_test = X_test.isnull().sum().sum()
na_y_train = y_train.isnull().sum()
na_y_test = y_test.isnull().sum()

print(f"NaN u X_train: {na_train}")
print(f"NaN u X_test:  {na_test}")
print(f"NaN u y_train: {na_y_train}")
print(f"NaN u y_test:  {na_y_test}")

# 3. Provera dimenzija.
print("\n" + "=" * 60)
print("DIMENZIJE PRIPREMLJENIH PODATAKA")
print("=" * 60)
print(f"X_train: {X_train.shape[0]} × {X_train.shape[1]}")
print(f"X_test:  {X_test.shape[0]} × {X_test.shape[1]}")
print(f"y_train: {y_train.shape[0]}")
print(f"y_test:  {y_test.shape[0]}")

# 4. Konzistentnost kolona.
if list(X_train.columns) == list(X_test.columns):
    print("\nKolone train i test se poklapaju.")
else:
    print("\nUPOZORENJE - kolone se ne poklapaju!")

PROVERA TIPOVA KOLONA
Tipovi kolona u X_train:
int64      20
float64     2
Name: count, dtype: int64

Sve kolone su numeričke.

PROVERA NEDOSTAJUĆIH VREDNOSTI
NaN u X_train: 0
NaN u X_test:  0
NaN u y_train: 0
NaN u y_test:  0

DIMENZIJE PRIPREMLJENIH PODATAKA
X_train: 5625 × 22
X_test:  1407 × 22
y_train: 5625
y_test:  1407

Kolone train i test se poklapaju.


### Prikaz finalnog `X_train`

Prikazujemo prvih 5 redova pripremljenog trening skupa da vizuelno
potvrdimo da je sve u numeričkom obliku i spremno za modele.

In [10]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,1,65,1,1,1,1,1,1,0,0,2,0,94.55,6078.75,1,0,1,0,0
1,1,0,0,0,26,0,0,0,0,1,1,0,0,0,0,35.75,1022.50,0,0,0,1,0
2,0,0,1,0,68,1,1,0,1,1,1,0,0,2,0,90.20,6297.65,1,0,1,0,0
3,1,0,0,0,3,1,0,0,1,0,0,0,1,0,0,84.30,235.05,1,0,0,1,0
4,0,0,1,0,49,0,0,1,0,0,0,1,0,0,0,40.65,2070.75,0,0,0,0,0


## 5. Čuvanje pripremljenih podataka

Pripremljene podatke čuvamo u folder `data/ml/`. Ovaj folder sadrži podatke **spremne za modeliranje** 

**Razlika između `data/processed/` i `data/ml/`:**
- `data/processed/` — podaci nakon split-a, **pre** enkodiranja (još čitljivi, "Yes"/"No")
- `data/ml/` — podaci **posle** enkodiranja (potpuno numerički, spremni za algoritme)

Ova podela omogućava da se u `data/processed/` može raditi EDA i vizualizacija
na čitljivim podacima, a `data/ml/` služi isključivo za modele.

In [11]:
putanja_ml = "../data/ml/"

# Čuvamo sva 4 objekta kao CSV.
X_train.to_csv(putanja_ml + "X_train.csv", index=False)
X_test.to_csv(putanja_ml + "X_test.csv", index=False)
y_train.to_csv(putanja_ml + "y_train.csv", index=False)
y_test.to_csv(putanja_ml + "y_test.csv", index=False)

print("Pripremljeni podaci sačuvani u folder 'data/ml/':")
print(f"  X_train.csv  ({X_train.shape[0]} x {X_train.shape[1]})")
print(f"  X_test.csv   ({X_test.shape[0]} x {X_test.shape[1]})")
print(f"  y_train.csv  ({y_train.shape[0]} vrednosti)")
print(f"  y_test.csv   ({y_test.shape[0]} vrednosti)")

Pripremljeni podaci sačuvani u folder 'data/ml/':
  X_train.csv  (5625 x 22)
  X_test.csv   (1407 x 22)
  y_train.csv  (5625 vrednosti)
  y_test.csv   (1407 vrednosti)


---

## 6. Zaključak

Uspešno smo pripremili podatke za mašinsko učenje kroz nekoliko faza:

**Feature engineering:**
1. Spojili `"No internet service"` i `"No phone service"` sa `"No"` u
   sedam kolona (jer je poslovno značenje isto)
2. Konvertovali `SeniorCitizen` iz brojnog (0/1) u tekstualni format
   (`"No"`/`"Yes"`) radi konzistentnosti

**Enkodiranje kategorija:**
- **Label Encoding (0/1)** primenjen na 13 binarnih kolona
- **Ordinal Encoding** primenjen na `Contract` (0 → 1 → 2, prati progresiju
  dužine ugovora)
- **One-Hot Encoding** primenjen na `InternetService` i `PaymentMethod`
  (sa `drop_first=True` radi izbegavanja kolinearnosti)
- Ciljna promenljiva `Churn` mapirana kao Yes → 1, No → 0

Broj kolona nakon obrade : 22

Sve kolone su numeričke (ili `int64` ili `float64`)

Nema nedostajućih vrednosti u bilo kom skupu

Train i test imaju identičan skup kolona
